## Anatomy of a Deployment manifest

A Deployment looks almost like a ReplicaSet — same `replicas`, `selector`, `template` — plus fields that govern *how the controller updates pods*.

```yaml
apiVersion: apps/v1
kind: Deployment
metadata: { name: web }
spec:
  replicas: 3
  selector:
    matchLabels: { app: web }
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 25%
      maxUnavailable: 25%
  revisionHistoryLimit: 10
  template:
    metadata:
      labels: { app: web }
    spec:
      containers:
        - name: app
          image: nginx:1.27-alpine
```

Field by field:

- **`replicas`** — target count, passed through to the active ReplicaSet.
- **`selector.matchLabels`** — must match the template labels, and is **immutable once set** — which avoids the orphaned-pods problem.
- **`template`** — the pod template. **The field you change most (usually the image); any change triggers a rollout.**
- **`strategy.type`** — `RollingUpdate` (default, gradual) or `Recreate` (kill all, then start all).
- **`maxSurge`** — extra pods allowed *above* `replicas` during a rollout (25% of 4 = 1 extra).
- **`maxUnavailable`** — how many desired pods may be down at once (25% of 4 = 1).
- **`revisionHistoryLimit`** — old ReplicaSets kept for rollback (default 10).
- **`minReadySeconds`** / **`progressDeadlineSeconds`** — extra confidence before "available"; fail a stalled rollout.

The mental split: fields you share with ReplicaSet describe *what to run*; `strategy` + its knobs describe *how to change it*. On our map, this manifest is the **Deployment** chip — and the rollout fields are what wire it to the **Rollout** box we reach next.